# Chapter 12: Django Web Apps, Part 1

Building the data models, admin site, and a simple home page.

*Adapted from the [Python Crash Course Cheat Sheets](https://ehmatthes.github.io/pcc/) by Eric Matthes, updated for modern Python and Google Colab.*


## What is Django?

Django is a web framework that helps you build interactive websites using Python. You define
the kind of data your site works with (**models**), and the ways users interact with that data
(**views** and **templates**).

> **Colab caveat:** Django expects a persistent project folder and a long-running development
> server (`manage.py runserver`) that you visit in a browser at `localhost:8000`. Colab's
> notebook cells run one at a time and don't expose `localhost` to your browser, so:
> * We use `!` to run Django's command-line tools directly in cells (this genuinely works).
> * We build a **real, complete project** on the Colab VM's disk, so all the code below is
>   correct and runnable.
> * To actually *view pages in a browser*, run this same project on your own computer (where
>   `localhost:8000` just works), or use a tunneling tool like `pyngrok` to expose Colab's
>   server temporarily. We call this out again at the end of Part 2.


### Installing Django


In [ ]:
!pip install -q django
import django
print("Django version:", django.get_version())


### Creating a project


In [ ]:
!django-admin startproject learning_log .
!ls


### Creating the database, and a superuser


In [ ]:
!python manage.py migrate


Creating a superuser interactively needs keyboard input, which doesn't work well in a notebook
cell. Instead, we create one non-interactively using Django's `shell` with environment
variables -- this is a common pattern for scripted/automated setups.


In [ ]:
import os

os.environ['DJANGO_SUPERUSER_USERNAME'] = 'admin'
os.environ['DJANGO_SUPERUSER_EMAIL'] = 'admin@example.com'
os.environ['DJANGO_SUPERUSER_PASSWORD'] = 'change-me-please'

!python manage.py createsuperuser --noinput


### Creating an app


In [ ]:
!python manage.py startapp learning_logs
!ls learning_logs


### Defining a model


In [ ]:
model_code = '''
from django.db import models


class Topic(models.Model):
    """A topic the user is learning about."""
    text = models.CharField(max_length=200)
    date_added = models.DateTimeField(auto_now_add=True)

    def __str__(self):
        return self.text
'''

with open('learning_logs/models.py', 'w') as f:
    f.write(model_code)

print(open('learning_logs/models.py').read())


### Activating the model, and migrating the database


In [ ]:
settings_path = 'learning_log/settings.py'
with open(settings_path) as f:
    settings_text = f.read()

if "'learning_logs'," not in settings_text:
    settings_text = settings_text.replace(
        "INSTALLED_APPS = [",
        "INSTALLED_APPS = [\n    'learning_logs',",
    )
    with open(settings_path, 'w') as f:
        f.write(settings_text)

print("learning_logs registered in INSTALLED_APPS.")


In [ ]:
!python manage.py makemigrations learning_logs
!python manage.py migrate


### Registering the model with the admin site


In [ ]:
admin_code = '''
from django.contrib import admin
from .models import Topic

admin.site.register(Topic)
'''

with open('learning_logs/admin.py', 'w') as f:
    f.write(admin_code)

print(open('learning_logs/admin.py').read())


## The Django shell

You can explore your project's data directly from the command line -- helpful for testing
queries before writing them into a view.


In [ ]:
shell_commands = '''
import django
django.setup()
from learning_logs.models import Topic
Topic.objects.create(text="Chess")
Topic.objects.create(text="Rock Climbing")
print(list(Topic.objects.all()))
'''

import subprocess
result = subprocess.run(
    ['python', 'manage.py', 'shell', '-c', shell_commands],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)


### Building a simple home page

A page needs a URL, a view, and a template. This is the shared pattern for every page you'll
add to a Django project.


In [ ]:
view_code = '''
from django.shortcuts import render


def index(request):
    """The home page for Learning Log."""
    return render(request, 'learning_logs/index.html')
'''

with open('learning_logs/views.py', 'w') as f:
    f.write(view_code)

import os
os.makedirs('learning_logs/templates/learning_logs', exist_ok=True)

with open('learning_logs/templates/learning_logs/index.html', 'w') as f:
    f.write('<p>Learning Log</p>\n<p>Learning Log helps you keep track of your learning, '
            'for any topic you\'re learning about.</p>\n')

urls_code = '''
from django.urls import path
from . import views

urlpatterns = [
    path('', views.index, name='index'),
]
'''
with open('learning_logs/urls.py', 'w') as f:
    f.write(urls_code)

# Wire the app's URLs into the project.
project_urls_path = 'learning_log/urls.py'
project_urls_code = '''
from django.contrib import admin
from django.urls import include, path

urlpatterns = [
    path('admin/', admin.site.urls),
    path('', include('learning_logs.urls')),
]
'''
with open(project_urls_path, 'w') as f:
    f.write(project_urls_code)

print("Home page wired up: URL -> view -> template.")


### Checking the project for obvious errors

`manage.py check` validates your project's configuration without starting a server -- a quick
sanity check that works fine in Colab.


In [ ]:
!python manage.py check


## Try it yourself

1. Add a second model, `Entry`, with a `ForeignKey` to `Topic` (we'll do this properly in Part
   2, but try sketching the class now).
2. Change the home page template's text to describe a project of your own choosing, then re-run
   `python manage.py check`.


In [ ]:
# Your code here
